# Deploying and Operating Machine Learning Models
---
## Now You Try: Track, Compare, and Serve a Candidate Model

You will create two MLflow runs, compare them, select a model to deploy, serve it locally, and identify how you would monitor it.


### Success criteria

By the end of the exercise, you should have:

- Two visible MLflow runs in one experiment.
- Parameters, metrics, and tags for both runs.
- At least one logged pipeline with an input example and signature.
- A reasoned choice of which model to deploy.
- A working local endpoint and successful prediction request.
- A simple monitoring and rollback plan.


### Exercise guide

1. Run the setup and data cells.
2. Complete the `fit_and_log_run()` function.
3. Log a baseline and a candidate run.
4. Compare the runs and select one.
5. Serve and test the selected model.
6. Complete the production-readiness reflection.


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import atexit
import socket
import subprocess

import numpy as np
import pandas as pd
import requests
import sklearn
import mlflow
import mlflow.sklearn

from IPython.display import Markdown, display
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")

print(f"Python: {sys.version.split()[0]}")
print(f"MLflow: {mlflow.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

In [ ]:
def find_data_file(filename: str) -> Path:
    """Find the course data whether the notebook is run from the lesson folder or a notebooks folder."""
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "data" / filename,
        Path.cwd().parent / "data" / filename,
        Path("/mnt/data") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")

DATA_PATH = find_data_file("SAheart.data")
WORK_DIR = (Path.cwd() / "mlops_artifacts").resolve()
MLRUNS_DIR = WORK_DIR / "mlruns"
LOG_DIR = WORK_DIR / "logs"

MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# A local file-based tracking store keeps this lesson self-contained.
TRACKING_URI = MLRUNS_DIR.as_uri()
mlflow.set_tracking_uri(TRACKING_URI)

print(f"Data: {DATA_PATH}")
print(f"Tracking store: {TRACKING_URI}")


In [ ]:
_BACKGROUND_PROCESSES = []


def port_is_open(port: int, host: str = "127.0.0.1") -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.25)
        return sock.connect_ex((host, port)) == 0


def wait_for_url(url: str, timeout: int = 30) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            response = requests.get(url, timeout=1)
            if response.status_code < 500:
                return True
        except requests.RequestException:
            pass
        time.sleep(0.5)
    return False


def start_background_process(command: list[str], log_name: str, env: dict | None = None):
    log_path = LOG_DIR / log_name
    log_file = open(log_path, "w", encoding="utf-8")
    process = subprocess.Popen(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env,
        text=True,
    )
    _BACKGROUND_PROCESSES.append((process, log_file))
    return process, log_path


def cleanup_background_processes():
    for process, log_file in _BACKGROUND_PROCESSES:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
        log_file.close()


atexit.register(cleanup_background_processes)


In [ ]:
heart = pd.read_csv(DATA_PATH, skipinitialspace=True)
heart["famhist"] = (heart["famhist"] == "Present").astype(int)

X = heart.drop(columns=["row.names", "chd"])
y = heart["chd"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=2022,
    stratify=y,
)

print(f"Rows: {len(heart):,}")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
X_train.head()


# Part 1: Complete the run-logging function

Fill each `TODO`. The function should:

- Build a pipeline.
- Fit it.
- Calculate accuracy and ROC AUC.
- Start an MLflow run.
- Log parameters, metrics, tags, and the model.
- Return a dictionary containing the run details.


In [ ]:
EXPERIMENT_NAME = "heart-disease-mlops-exercise"
mlflow.set_experiment(EXPERIMENT_NAME)


def fit_and_log_run(
    *,
    run_name: str,
    C: float,
    penalty: str,
    class_weight=None,
) -> dict:
    # TODO 1: Create a pipeline with StandardScaler and LogisticRegression.
    # Use solver="liblinear", max_iter=2_000, and random_state=2022.
    model = None

    # TODO 2: Fit the model and calculate predictions and probabilities.
    predictions = None
    probabilities = None

    # TODO 3: Calculate accuracy and ROC AUC.
    accuracy = None
    roc_auc = None

    input_example = X_test.iloc[:3].copy()

    # TODO 4: Infer a signature from the input example and its predictions.
    signature = None

    # TODO 5: Start a named MLflow run and log parameters, metrics, and tags.
    # Then log the complete pipeline with name="heart_disease_pipeline".
    run_id = None
    model_uri = None

    return {
        "run_name": run_name,
        "run_id": run_id,
        "model_uri": model_uri,
        "accuracy": accuracy,
        "roc_auc": roc_auc,
        "C": C,
        "penalty": penalty,
        "class_weight": class_weight,
    }


## Checkpoint

Before continuing, confirm that the function:

- Returns a fitted model artifact URI.
- Logs numeric metrics rather than formatted strings.
- Logs the complete pipeline, not just the logistic regression step.
- Uses an input example with the same columns and data types expected at inference time.


# Part 2: Log two candidate runs

Run the baseline configuration first. Then change at least one meaningful setting for the candidate.

Suggested candidate choices:

- Change `C` to alter regularization strength.
- Change from `l1` to `l2` penalty.
- Add `class_weight="balanced"`.


In [ ]:
baseline_result = fit_and_log_run(
    run_name="baseline",
    C=0.123,
    penalty="l1",
    class_weight=None,
)

# TODO 6: Change at least one setting for your candidate run.
candidate_result = fit_and_log_run(
    run_name="candidate",
    C=1.0,
    penalty="l2",
    class_weight="balanced",
)

results = pd.DataFrame([baseline_result, candidate_result])
results[["run_name", "accuracy", "roc_auc", "C", "penalty", "class_weight"]]


# Part 3: Select a model to deploy

Choose the model using more than one consideration. For this exercise, ROC AUC is the primary metric, but you should also consider:

- Accuracy.
- Difference in configuration complexity.
- Whether class weighting is appropriate for the use case.
- Whether the improvement is large enough to justify changing the model.


In [ ]:
# TODO 7: Replace this rule when you have made your decision.
selected_result = max(
    [baseline_result, candidate_result],
    key=lambda result: result["roc_auc"],
)

model_uri = selected_result["model_uri"]
print(f"Selected run: {selected_result['run_name']}")
print(f"ROC AUC: {selected_result['roc_auc']:.3f}")
print(f"Model URI: {model_uri}")


**Write your deployment decision:**

- Selected model:
- Evidence:
- Trade-off accepted:
- What additional evidence would you want before a real release?


# Part 4: Inspect the runs

Start the MLflow UI and compare the two runs side by side.


In [ ]:
UI_PORT = 5000
UI_URL = f"http://127.0.0.1:{UI_PORT}"

env = os.environ.copy()
env["MLFLOW_TRACKING_URI"] = TRACKING_URI

if port_is_open(UI_PORT):
    print(f"Port {UI_PORT} is already in use. Open {UI_URL} and check whether MLflow is running there.")
else:
    ui_command = [
        sys.executable,
        "-m",
        "mlflow",
        "ui",
        "--backend-store-uri",
        TRACKING_URI,
        "--host",
        "127.0.0.1",
        "--port",
        str(UI_PORT),
    ]
    ui_process, ui_log = start_background_process(ui_command, "mlflow-ui.log", env=env)
    if wait_for_url(UI_URL, timeout=30):
        print(f"MLflow UI is running at {UI_URL}")
    else:
        print(f"MLflow UI did not start. Inspect the log: {ui_log}")


Open **http://127.0.0.1:5000** and check:

- Can you distinguish the runs from their names and tags?
- Are all model settings visible?
- Are the metrics directly comparable?
- Does each model include a signature and input example?


# Part 5: Serve the selected model

The next cell starts the selected model on port `5001`.


In [ ]:
MODEL_PORT = 5001
MODEL_BASE_URL = f"http://127.0.0.1:{MODEL_PORT}"

if port_is_open(MODEL_PORT):
    raise RuntimeError(
        f"Port {MODEL_PORT} is already in use. Stop the existing process or change MODEL_PORT."
    )

env = os.environ.copy()
env["MLFLOW_TRACKING_URI"] = TRACKING_URI

serve_command = [
    sys.executable,
    "-m",
    "mlflow",
    "models",
    "serve",
    "--model-uri",
    model_uri,
    "--host",
    "127.0.0.1",
    "--port",
    str(MODEL_PORT),
    "--env-manager",
    "local",
]

serve_process, serve_log = start_background_process(
    serve_command,
    "mlflow-model-server.log",
    env=env,
)

if wait_for_url(f"{MODEL_BASE_URL}/health", timeout=60):
    print(f"Model server is healthy at {MODEL_BASE_URL}")
else:
    raise RuntimeError(f"Model server did not start. Inspect the log: {serve_log}")


## Send a prediction request

Complete the payload using two test rows and send it to `/invocations`.


In [ ]:
request_rows = X_test.iloc[:2].copy()

# TODO 8: Create a dataframe_split payload.
payload = None

# TODO 9: Send the POST request and print the status code and JSON response.
response = None


# Part 6: Production-readiness reflection

Complete the table before finishing.

| Question | Your answer |
|---|---|
| Which service-health metric would you monitor? | |
| Which data-quality or drift signal would you monitor? | |
| Which model or business outcome would you monitor? | |
| What would trigger an alert? | |
| What would trigger rollback or retraining? | |
| Who should own the response? | |


## Optional stretch: test the model contract

Send one invalid request by removing or renaming a feature. Record:

- The HTTP status code.
- The error message.
- Why rejecting the request is safer than silently generating a prediction.


## Cleanup


In [ ]:
cleanup_background_processes()
print("Background MLflow processes stopped.")
